In [11]:
import torch, platform
print("Python:", platform.python_version())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version (torch):", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))


Python: 3.12.9
Torch: 2.5.1
CUDA available: True
CUDA version (torch): 11.8
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [1]:
import torch

print("is_available:", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count())

# Force CUDA init + a real op
x = torch.randn(10, device="cuda")
print("cuda tensor ok:", x[:3])
print("GPU:", torch.cuda.get_device_name(0))


is_available: True
device_count: 1
cuda tensor ok: tensor([-1.5878,  0.3930,  1.2913], device='cuda:0')
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
# ============================================================
# GROKKING DEPTH EXPERIMENT (UPDATED — GROKKING RELIABLE)
# Does deeper network grok faster?
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from itertools import product
import time
import pandas as pd

# ============================================================
# Configuration (UPDATED FOR RELIABLE GROKKING)
# ============================================================

P = 97
TRAIN_FRACTION = 0.2     # critical
STEPS = 400_000          # allow grokking transition
LR = 1e-2                # required for SGD grokking
WEIGHT_DECAY = 2e-3      # critical strength
MOMENTUM = 0.9
SEEDS = 5
LOG_EVERY = 2000

DEPTHS = [2, 4, 8]
WIDTH = 256              # improved capacity

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ============================================================
# Reproducibility
# ============================================================

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True

# ============================================================
# Dataset
# ============================================================

def generate_modular_addition_data(p=97, train_fraction=0.2, seed=0):

    random.seed(seed)

    pairs = list(product(range(p), range(p)))
    random.shuffle(pairs)

    split_idx = int(len(pairs) * train_fraction)

    train_pairs = pairs[:split_idx]
    test_pairs = pairs[split_idx:]

    def to_tensor(data):

        x = torch.tensor(data, dtype=torch.long).to(DEVICE)
        y = (x[:, 0] + x[:, 1]) % p

        return x, y.to(DEVICE)

    return to_tensor(train_pairs), to_tensor(test_pairs)

# ============================================================
# Model
# ============================================================

class ModularMLP(nn.Module):

    def __init__(self, p=97, width=256, depth=4):

        super().__init__()

        self.embed = nn.Embedding(p, width)

        layers = []

        for _ in range(depth):
            layers.append(nn.Linear(width, width))
            layers.append(nn.GELU())

        self.mlp = nn.Sequential(*layers)

        self.output = nn.Linear(width, p)

    def forward(self, x):

        a = self.embed(x[:, 0])
        b = self.embed(x[:, 1])

        h = a + b
        h = self.mlp(h)

        return self.output(h)

# ============================================================
# Accuracy
# ============================================================

def accuracy(logits, targets):

    preds = logits.argmax(dim=-1)

    return (preds == targets).float().mean().item()

# ============================================================
# Training
# ============================================================

def train_model(depth, seed):

    set_seed(seed)

    (train_x, train_y), (test_x, test_y) = generate_modular_addition_data(
        p=P,
        train_fraction=TRAIN_FRACTION,
        seed=seed
    )

    model = ModularMLP(p=P, width=WIDTH, depth=depth).to(DEVICE)

    # CRITICAL: SGD optimizer
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=LR,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY
    )

    T_train_acc = None
    T_test_acc = None

    start_time = time.time()

    for step in range(STEPS):

        logits = model(train_x)

        loss = F.cross_entropy(logits, train_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % LOG_EVERY == 0:

            model.eval()

            with torch.no_grad():

                train_acc = accuracy(model(train_x), train_y)
                test_acc = accuracy(model(test_x), test_y)

            if train_acc >= 0.99 and T_train_acc is None:
                T_train_acc = step

            if test_acc >= 0.99 and T_test_acc is None:
                T_test_acc = step

            model.train()

    delay = None

    if T_train_acc is not None and T_test_acc is not None:

        delay = T_test_acc - T_train_acc

    total_time = round(time.time() - start_time, 2)

    return {
        "depth": depth,
        "seed": seed,
        "T_train": T_train_acc,
        "T_test": T_test_acc,
        "delay": delay,
        "time_sec": total_time
    }

# ============================================================
# Run Experiment
# ============================================================

results = []

for depth in DEPTHS:

    print(f"\n==============================")
    print(f"Running Depth = {depth}")
    print(f"==============================")

    for seed in range(SEEDS):

        res = train_model(depth, seed)

        results.append(res)

        print(res)

df = pd.DataFrame(results)

print("\n====================================")
print("SUMMARY BY DEPTH")
print("====================================")

print(df.groupby("depth")["delay"].mean())

print("\nFull Results:")
print(df)


Using device: cuda

Running Depth = 2
{'depth': 2, 'seed': 0, 'T_train': 2000, 'T_test': 34000, 'delay': 32000, 'time_sec': 1071.78}
{'depth': 2, 'seed': 1, 'T_train': 2000, 'T_test': 22000, 'delay': 20000, 'time_sec': 1237.62}
{'depth': 2, 'seed': 2, 'T_train': 2000, 'T_test': 28000, 'delay': 26000, 'time_sec': 1677.5}
{'depth': 2, 'seed': 3, 'T_train': 2000, 'T_test': None, 'delay': None, 'time_sec': 1777.89}
{'depth': 2, 'seed': 4, 'T_train': 2000, 'T_test': 212000, 'delay': 210000, 'time_sec': 1812.32}

Running Depth = 4
{'depth': 4, 'seed': 0, 'T_train': None, 'T_test': None, 'delay': None, 'time_sec': 2469.69}
{'depth': 4, 'seed': 1, 'T_train': None, 'T_test': None, 'delay': None, 'time_sec': 2502.52}
{'depth': 4, 'seed': 2, 'T_train': None, 'T_test': None, 'delay': None, 'time_sec': 1805.85}
{'depth': 4, 'seed': 3, 'T_train': None, 'T_test': None, 'delay': None, 'time_sec': 1776.8}
{'depth': 4, 'seed': 4, 'T_train': None, 'T_test': None, 'delay': None, 'time_sec': 1826.49}

Runn

KeyboardInterrupt: 

In [6]:
# ============================================================
# GROKKING (DEPTH = 8) — STABLE VERSION (NO DIVERGENCE / NaNs)
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from itertools import product
import time
import pandas as pd

# ============================================================
# Config
# ============================================================

P = 97
TRAIN_FRACTION = 0.2

DEPTH = 8
WIDTH = 512

STEPS = 800_000
LOG_EVERY = 2000
SEEDS = 3

# FIXES vs your diverging run:
LR = 3e-2              # ↓ from 1e-1
WEIGHT_DECAY = 1e-3
MOMENTUM = 0.9
CLIP_NORM = 1.0        # NEW: prevents weight explosion

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA (torch):", torch.version.cuda)

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

# ============================================================
# Reproducibility
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.random.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

# ============================================================
# Data
# ============================================================

def generate_modular_addition_data(p=97, train_fraction=0.2, seed=0):
    random.seed(seed)
    pairs = list(product(range(p), range(p)))
    random.shuffle(pairs)

    split_idx = int(len(pairs) * train_fraction)
    train_pairs = pairs[:split_idx]
    test_pairs = pairs[split_idx:]

    def to_tensor(data):
        x = torch.tensor(data, dtype=torch.long, device=DEVICE)
        y = (x[:, 0] + x[:, 1]) % p
        return x, y.to(DEVICE)

    return to_tensor(train_pairs), to_tensor(test_pairs)

# ============================================================
# Model
# ============================================================

class ResidualBlock(nn.Module):
    def __init__(self, width: int):
        super().__init__()
        self.ln1 = nn.LayerNorm(width)
        self.fc1 = nn.Linear(width, width)
        self.ln2 = nn.LayerNorm(width)
        self.fc2 = nn.Linear(width, width)
        self.act = nn.GELU()

    def forward(self, x):
        h = self.ln1(x)
        h = self.act(self.fc1(h))
        h = self.ln2(h)
        h = self.fc2(self.act(h))
        return x + 0.5 * h

class ModularResMLP(nn.Module):
    def __init__(self, p=97, width=512, depth=8):
        super().__init__()
        self.embed = nn.Embedding(p, width)
        self.blocks = nn.Sequential(*[ResidualBlock(width) for _ in range(depth)])
        self.ln_out = nn.LayerNorm(width)
        self.output = nn.Linear(width, p)

    def forward(self, x):
        a = self.embed(x[:, 0])
        b = self.embed(x[:, 1])
        h = a + b
        h = self.blocks(h)
        h = self.ln_out(h)
        return self.output(h)

# ============================================================
# Metrics
# ============================================================

def accuracy(logits, targets):
    return (logits.argmax(dim=-1) == targets).float().mean().item()

# ============================================================
# Train
# ============================================================

def train_depth8(seed: int):
    set_seed(seed)
    (train_x, train_y), (test_x, test_y) = generate_modular_addition_data(
        p=P, train_fraction=TRAIN_FRACTION, seed=seed
    )

    model = ModularResMLP(p=P, width=WIDTH, depth=DEPTH).to(DEVICE)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=LR,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
    )

    T_train_acc = None
    T_test_acc = None
    start = time.time()

    for step in range(STEPS):
        logits = model(train_x)
        loss = F.cross_entropy(logits, train_y)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()

        # NEW: stabilize updates
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CLIP_NORM)

        optimizer.step()

        # Abort early if NaNs appear (saves time)
        if not torch.isfinite(loss).item():
            print(f"NaN/Inf loss detected at step={step}. Aborting seed={seed}.")
            break

        if step % LOG_EVERY == 0:
            model.eval()
            with torch.no_grad():
                train_acc = accuracy(model(train_x), train_y)
                test_acc = accuracy(model(test_x), test_y)
                w_norm = float(model.output.weight.norm())

            elapsed_min = (time.time() - start) / 60
            print(
                f"seed={seed} step={step}/{STEPS} "
                f"loss={loss.item():.4f} train={train_acc:.3f} test={test_acc:.3f} "
                f"w_norm={w_norm:.2f} elapsed={elapsed_min:.1f}m"
            )

            if train_acc >= 0.99 and T_train_acc is None:
                T_train_acc = step
            if test_acc >= 0.99 and T_test_acc is None:
                T_test_acc = step

            model.train()

            # Stop early once grokking is confirmed
            if T_train_acc is not None and T_test_acc is not None:
                break

    delay = None
    if T_train_acc is not None and T_test_acc is not None:
        delay = T_test_acc - T_train_acc

    return {
        "depth": DEPTH,
        "seed": seed,
        "T_train": T_train_acc,
        "T_test": T_test_acc,
        "delay": delay,
        "time_sec": round(time.time() - start, 2),
    }

# ============================================================
# Run
# ============================================================

print(f"\n==============================")
print(f"Running Depth = {DEPTH} only")
print(f"==============================")

results = []
for seed in range(SEEDS):
    res = train_depth8(seed)
    results.append(res)
    print("RESULT:", res)

df = pd.DataFrame(results)

print("\n====================================")
print("SUMMARY (DEPTH 8)")
print("====================================")
print(df[["seed", "T_train", "T_test", "delay", "time_sec"]])
print("\nMean delay (non-null):", df["delay"].dropna().mean())


Using device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
CUDA (torch): 11.8

Running Depth = 8 only
seed=0 step=0/800000 loss=4.7300 train=0.010 test=0.009 w_norm=5.69 elapsed=0.0m
seed=0 step=2000/800000 loss=0.0120 train=1.000 test=0.199 w_norm=10.64 elapsed=0.4m
seed=0 step=4000/800000 loss=0.0101 train=1.000 test=0.199 w_norm=9.83 elapsed=0.9m
seed=0 step=6000/800000 loss=0.0172 train=1.000 test=0.201 w_norm=9.01 elapsed=1.3m
seed=0 step=8000/800000 loss=0.0285 train=1.000 test=0.203 w_norm=7.63 elapsed=1.7m
seed=0 step=10000/800000 loss=0.1498 train=0.998 test=0.208 w_norm=6.16 elapsed=2.1m
seed=0 step=12000/800000 loss=0.0022 train=1.000 test=0.226 w_norm=5.41 elapsed=2.6m
seed=0 step=14000/800000 loss=0.0024 train=1.000 test=0.239 w_norm=5.04 elapsed=3.0m
seed=0 step=16000/800000 loss=0.0021 train=1.000 test=0.268 w_norm=4.95 elapsed=3.4m
seed=0 step=18000/800000 loss=0.0023 train=1.000 test=0.295 w_norm=4.84 elapsed=3.8m
seed=0 step=20000/800000 loss=0.0021 train=1.000 test=0